# Properties of the polar Fourier transform

The polar Fourier transform (PFT) is linear, so on a fixed grid it is a matrix. Baddour defines the discrete 2-D Fourier transform in polar coordinates directly as that matrix, its kernel, and derives the rules it obeys from the kernel alone [[1]](#ref-1): orthogonality, a delta pair, a generalized shift with its modulation, convolution, and multiplication rules, and a Parseval relation. This notebook builds the kernel from its definition, checks that `pypft.forward_pft` and `pypft.inverse_pft` apply exactly that kernel, and then checks each rule, followed by four properties of the transform that are easiest to see on whole signals: rotation, linearity, the zeroth harmonic, and the symmetry of a real signal's spectrum.

The properties of the discrete Hankel transform alone are covered in [03_discrete_hankel_transform](03_discrete_hankel_transform.ipynb).

In [ ]:
# NumPy for arrays, matplotlib for figures, SciPy for Bessel functions, and PyPFT itself
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import jn_zeros, jv

import pypft

## The kernels $E^-$ and $E^+$

A grid has $N_2 = 2M + 1$ angular samples, indexed $p, q = -M, \dots, M$, and $N_1 - 1$ radial samples, indexed $k, m = 1, \dots, N_1 - 1$. A space-domain sample is $f_{pk}$ and a frequency-domain sample $F_{qm}$. The forward and inverse transforms are [[1]](#ref-1) (Eqs. 39-40)

$$F_{qm} = \sum_{k=1}^{N_1-1} \sum_{p=-M}^{M} E^{-}_{qm;pk}\, f_{pk}, \qquad f_{pk} = \sum_{m=1}^{N_1-1} \sum_{q=-M}^{M} E^{+}_{qm;pk}\, F_{qm},$$

with the forward kernel $E^-$ and the inverse kernel $E^+$ [[1]](#ref-1) (Eq. 32)

$$E^{-}_{qm;pk} = \frac{1}{N_2} \sum_{n=-M}^{M} i^{-n}\, \frac{2\, J_n\!\left(j_{nk}\, j_{nm} / j_{nN_1}\right)}{j_{nN_1}^2\, J_{n+1}^2(j_{nk})}\, e^{-i n 2\pi p / N_2}\, e^{+i n 2\pi q / N_2},$$

$$E^{+}_{qm;pk} = \frac{1}{N_2} \sum_{n=-M}^{M} i^{n}\, \frac{2\, J_n\!\left(j_{nk}\, j_{nm} / j_{nN_1}\right)}{J_{n+1}^2(j_{nm})}\, e^{+i n 2\pi p / N_2}\, e^{-i n 2\pi q / N_2}.$$

Each term of the sum is one harmonic $n$: an angular phase factor, an inverse DFT in one direction and a DFT in the other, times the order-$n$ DHT kernel of [03_discrete_hankel_transform](03_discrete_hankel_transform.ipynb) and the factor $i^{\mp n}$ of the 2-D Fourier transform. Summing over $n$ composes exactly the three steps of the chain into one matrix. $j_{nk}$ is the $k$-th positive zero of $J_n$, which for negative $n$ is the zero of $J_{|n|}$.

`forward_pft` and `inverse_pft` add the physical scale of a grid with space limit $R$: `forward_pft` applies $2\pi R^2 E^-$ and `inverse_pft` applies $E^+ / (2\pi R^2)$.

The helper below builds both kernels from the definition. A pair of indices $(m, q)$ or $(k, p)$ becomes one flat index in the same order as `values.ravel()` on PyPFT's `(radial, angular)` layout, so each kernel is an $(N_1 - 1) N_2 \times (N_1 - 1) N_2$ matrix with frequency rows and space columns.

In [ ]:
def pft_kernels(grid):
    """Build E^- and E^+ of a grid from their definition (rows qm, columns pk)."""
    n_radial, n_angular = grid.n_radial, grid.n_angular
    size = n_radial * n_angular
    E_minus = np.zeros(shape=(size, size), dtype=complex)
    E_plus = np.zeros(shape=(size, size), dtype=complex)

    # The angular indices -M..M, which are also the harmonic orders
    p = grid.harmonics
    for n in grid.harmonics:
        n = int(n)
        # The first N_1 zeros of J_|n|: j_k for k = 1..N_1 - 1, and j_N
        zeros = jn_zeros(n=abs(n), nt=n_radial + 1)
        j_k, j_N = zeros[:-1], zeros[-1]

        # The DHT factor (rows m, columns k), scaled by column for E^-, by row for E^+
        bessel = 2 * jv(n, np.outer(a=j_k, b=j_k) / j_N)
        radial_minus = bessel / (j_N**2 * jv(n + 1, j_k)[np.newaxis, :] ** 2)
        radial_plus = bessel / jv(n + 1, j_k)[:, np.newaxis] ** 2

        # The angular factor exp(+i n 2 pi q / N_2) exp(-i n 2 pi p / N_2), rows q
        angular = np.exp(1j * n * 2 * np.pi * np.subtract.outer(p, p) / n_angular)

        # The radial index is the slow one in the flat index, so it goes first
        E_minus += 1j ** (-n) * np.kron(a=radial_minus, b=angular) / n_angular
        E_plus += 1j**n * np.kron(a=radial_plus, b=angular.conj()) / n_angular
    return E_minus, E_plus


# A small grid: 48 radial and 31 angular samples within R = 8
grid = pypft.PolarGrid(n_radial=48, n_angular=31, R=8.0)
size = grid.n_radial * grid.n_angular
E_minus, E_plus = pft_kernels(grid=grid)

# forward_pft and inverse_pft of every basis vector at once, as a batch of size signals
basis = np.eye(N=size, dtype=complex).reshape(grid.n_radial, grid.n_angular, size)
forward_matrix = pypft.forward_pft(f=basis, grid=grid).reshape(size, size)
inverse_matrix = pypft.inverse_pft(F=basis, grid=grid).reshape(size, size)

# They are the kernels with the physical scale
scale = 2 * np.pi * grid.R**2
forward_mismatch = np.abs(forward_matrix - scale * E_minus).max()
inverse_mismatch = np.abs(inverse_matrix - E_plus.T / scale).max()
print(f"kernel size: {E_minus.shape}")
print(
    f"forward_pft vs. 2 pi R^2 E^-:  {forward_mismatch / np.abs(scale * E_minus).max():.1e}"
)
print(
    f"inverse_pft vs. E^+ / 2 pi R^2: {inverse_mismatch / np.abs(E_plus / scale).max():.1e}"
)

## Orthogonality

The two kernels are inverses of each other. Summing their product over the frequency indices gives a delta in the space indices, and summing over the space indices a delta in the frequency indices [[1]](#ref-1) (Eqs. 34, 37):

$$\sum_{m=1}^{N_1-1} \sum_{q=-M}^{M} E^{-}_{qm;pk}\, E^{+}_{qm;p'k'} = \delta_{pp'}\, \delta_{kk'}, \qquad \sum_{k=1}^{N_1-1} \sum_{p=-M}^{M} E^{-}_{qm;pk}\, E^{+}_{q'm';pk} = \delta_{qq'}\, \delta_{mm'}.$$

As matrices, $(E^-)^T E^+ = I$ and $E^- (E^+)^T = I$, which is why the inverse transform undoes the forward one. Like the DHT's self-inverse relation, this holds up to a small residual that shrinks as the grid grows.

The kernels are not symmetric: $E^-$ divides by $j_{nN_1}^2 J_{n+1}^2(j_{nk})$, a function of the space index $k$, and $E^+$ by $J_{n+1}^2(j_{nm})$, a function of the frequency index $m$. So $E^+$ is not the complex conjugate of $E^-$, and $E^-$ is not unitary: $E^- (E^-)^H$, with $(E^-)^H$ the conjugate transpose, is not the identity, not even up to a constant. The figure shows the top-left corner of both products, the first three radial indices with all their angular indices, each divided by its largest entry: $E^- (E^+)^T$ is the identity, $E^- (E^-)^H$ is not.

In [ ]:
# Both orthogonality relations
identity = np.eye(N=size)
print(f"max |(E^-)^T E^+ - I|: {np.abs(E_minus.T @ E_plus - identity).max():.1e}")
print(f"max |E^- (E^+)^T - I|: {np.abs(E_minus @ E_plus.T - identity).max():.1e}")

# The corner of both products covering the first three radial indices
corner = 3 * grid.n_angular
products = (
    (E_minus @ E_plus.T, "$|E^- (E^+)^T|$"),
    (E_minus @ E_minus.conj().T, "$|E^- (E^-)^H|$"),
)

# Each corner divided by its largest entry, on the same color scale
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(11, 4.5))
for ax, (product, title) in zip(axes, products):
    block = np.abs(product[:corner, :corner])
    shown = ax.imshow(X=block / block.max(), vmin=0, vmax=1, cmap="magma")
    ax.set_xlabel(xlabel="flat index")
    ax.set_ylabel(ylabel="flat index")
    ax.set_title(label=f"{title}, divided by its maximum")
    fig.colorbar(mappable=shown, ax=ax)
fig.tight_layout()
plt.show()

## The delta pair

Orthogonality gives a transform pair at once. The space-domain signal $f_{pk} = E^{+}_{q_0 m_0;pk}$, one row of the inverse kernel, transforms into a delta at $(q_0, m_0)$ [[1]](#ref-1) (Eq. 43):

$$E^{+}_{q_0 m_0;pk} \;\Longleftrightarrow\; \delta_{qq_0}\, \delta_{mm_0}.$$

It is the discrete counterpart of a complex exponential transforming into a delta. Up to the physical scale, the same signal is `inverse_pft` of a delta. The figure shows that signal and its transform on the polar index table.

In [ ]:
# The flat index of a pair (radial index, angular index), both counted from 0
def flat(radial, angular):
    return radial * grid.n_angular + angular


# One row of E^+, reshaped onto the (radial, angular) layout
m0, q0 = 6, 20
row = E_plus[flat(radial=m0, angular=q0)].reshape(grid.n_radial, grid.n_angular)

# Its forward transform is a delta at (m0, q0)
transformed = (E_minus @ row.ravel()).reshape(grid.n_radial, grid.n_angular)
delta = np.zeros_like(a=transformed)
delta[m0, q0] = 1.0
print(f"max |transform - delta|: {np.abs(transformed - delta).max():.1e}")

# The row and its transform, as magnitude and phase
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 8))
pypft.plot_signal(
    signal=pypft.PolarSpatialSignal(values=row, grid=grid), ax=(axes[0, 0], axes[0, 1])
)
pypft.plot_signal(
    signal=pypft.PolarFrequencySignal(values=transformed, grid=grid),
    ax=(axes[1, 0], axes[1, 1]),
)
fig.tight_layout()
plt.show()

## The generalized shift

A 2-D Fourier transform turns a translation into a modulation. On a polar grid, a translated sample generally does not land on another sample, so a shift cannot be defined by moving indices. As for the DHT, Baddour defines the shift through the transform: transform, multiply by the kernel evaluated at the shift point $(p_0, k_0)$, transform back [[1]](#ref-1) (Eq. 46):

$$f^{p_0 k_0}_{pk} = \sum_{m=1}^{N_1-1} \sum_{q=-M}^{M} F_{qm}\, E^{-}_{qm;p_0 k_0}\, E^{+}_{qm;pk}.$$

It behaves like a translation. A narrow bump at the center, shifted by $(p_0, k_0)$, comes back as a bump at the sample point of angular index $p_0$ and radial index $k_0$, marked with a cross in the figure, plus a faint ring at the same radius.

In [ ]:
def shift_space(values, index0):
    """Shift a flattened space-domain signal by the flat index index0 (Eq. 46)."""
    return E_plus.T @ (E_minus[:, index0] * (E_minus @ values))


# A narrow bump at the center of the disk
bump = np.exp(-((grid.r.T / 0.4) ** 2)).astype(dtype=complex)

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 4.5))
for ax, (p0, k0) in zip(axes, ((15, 0), (20, 15), (5, 30))):
    # Shift the bump and render it in the plane
    shifted = shift_space(values=bump.ravel(), index0=flat(radial=k0, angular=p0))
    shifted_signal = pypft.PolarSpatialSignal(
        values=shifted.reshape(grid.n_radial, grid.n_angular), grid=grid
    )
    pypft.render_cartesian(signal=shifted_signal, height=300, width=300, ax=ax)

    # Mark the sample point of the shift (y down, as in render_cartesian)
    r0, theta0 = grid.r[p0, k0], grid.theta[p0]
    ax.plot(
        r0 * np.cos(theta0),
        -r0 * np.sin(theta0),
        marker="+",
        color="red",
        markersize=12,
    )
    ax.set_title(label=f"shifted by $(p_0, k_0) = ({p0}, {k0})$")
fig.tight_layout()
plt.show()

## Rules built on the generalized shift

With the generalized shift, the PFT has the same family of rules as the ordinary Fourier transform. Writing $f \Leftrightarrow F$ for a transform pair [[1]](#ref-1) (Eqs. 53, 62, 68, 71):

- **Shift-modulation.** A shift in space is a modulation of the transform by a column of $E^-$:

  $$f^{p_0 k_0}_{pk} \;\Longleftrightarrow\; E^{-}_{qm;p_0 k_0}\, F_{qm}.$$
- **Modulation.** A modulation in space by a row of $E^+$ is a shift of the transform, the shift in frequency being defined like the shift in space, with the roles of the kernels swapped [[1]](#ref-1) (Eq. 60):

  $$E^{+}_{q_0 m_0;pk}\, g_{pk} \;\Longleftrightarrow\; G^{q_0 m_0}_{qm}.$$
- **Convolution.** The convolution built on the generalized shift transforms into a product:

  $$(h * g)_{pk} = \sum_{k_0=1}^{N_1-1} \sum_{p_0=-M}^{M} h^{p_0 k_0}_{pk}\, g_{p_0 k_0} \;\Longleftrightarrow\; H_{qm}\, G_{qm}.$$
- **Multiplication.** A product in space transforms into a convolution of the transforms:

  $$h_{pk}\, g_{pk} \;\Longleftrightarrow\; (H * G)_{qm} = \sum_{m'=1}^{N_1-1} \sum_{q'=-M}^{M} H_{q'm'}\, G^{q'm'}_{qm}.$$

The cell checks all four with random signals, each rule's two sides computed independently. The convolution and multiplication need every shift at once, a matrix whose column $i_0$ is the shift by $i_0$.

In [ ]:
def shift_frequency(values, index0):
    """Shift a flattened frequency-domain signal by the flat index index0 (Eq. 60)."""
    return E_minus @ (E_plus[index0] * (E_plus.T @ values))


# Two random complex signals and their transforms
rng = np.random.default_rng(seed=0)
g = rng.standard_normal(size=size) + 1j * rng.standard_normal(size=size)
h = rng.standard_normal(size=size) + 1j * rng.standard_normal(size=size)
G, H = E_minus @ g, E_minus @ h
index0 = flat(radial=10, angular=4)

# Shift-modulation: the transform of a shifted signal
shift_modulation = (
    E_minus @ shift_space(values=g, index0=index0) - E_minus[:, index0] * G
)

# Modulation: the transform of a modulated signal
modulation = E_minus @ (E_plus[index0] * g) - shift_frequency(values=G, index0=index0)

# Convolution: column i0 of the first matrix is h shifted by i0 in space
h_shifts = E_plus.T @ (E_minus * H[:, np.newaxis])
convolution = E_minus @ (h_shifts @ g) - H * G

# Multiplication: column i0 of the second matrix is G shifted by i0 in frequency
G_shifts = E_minus @ (E_plus.T * g[:, np.newaxis])
multiplication = E_minus @ (h * g) - G_shifts @ H

for name, residual in (
    ("shift-modulation", shift_modulation),
    ("modulation", modulation),
    ("convolution", convolution),
    ("multiplication", multiplication),
):
    print(f"{name:>16}: max residual {np.abs(residual).max():.1e}")

`G_shifts` uses `E_plus.T @ G = g`, so its columns are the frequency shifts of `G` as defined above.

## Parseval's relation

For the ordinary DFT, a unitary transform, the energy $\sum \lvert h \rvert^2$ is the same in both domains. $E^-$ is not unitary, so the plain sums differ. Baddour's fix is a **starred conjugate** built with the opposite kernel [[1]](#ref-1) (Eqs. 80, 83):

$$\overline{h}^{*}_{pk} = \sum_{m=1}^{N_1-1} \sum_{q=-M}^{M} \overline{H_{qm}}\, E^{-}_{qm;pk}, \qquad \overline{H}^{*}_{qm} = \sum_{k=1}^{N_1-1} \sum_{p=-M}^{M} \overline{h_{pk}}\, E^{+}_{qm;pk}.$$

With it, energy is preserved in the form [[1]](#ref-1) (Eq. 88)

$$\sum_{m,q} \lvert H_{qm} \rvert^2 = \sum_{k,p} h_{pk}\, \overline{h}^{*}_{pk}, \qquad \sum_{k,p} \lvert h_{pk} \rvert^2 = \sum_{m,q} H_{qm}\, \overline{H}^{*}_{qm}.$$

Each is a one-line consequence of orthogonality: the starred conjugate undoes the transform on the other side of the sum.

In [ ]:
# The plain energies of h and of its transform H differ
energy_space = np.sum(a=np.abs(h) ** 2)
energy_frequency = np.sum(a=np.abs(H) ** 2)
print(f"sum |h|^2: {energy_space:.6e}")
print(f"sum |H|^2: {energy_frequency:.6e}")

# The starred conjugates, each built with the opposite kernel
h_star = E_minus.T @ np.conj(H)
H_star = E_plus @ np.conj(h)

# Each energy equals the starred sum on the other side
print(f"sum h h*:  {np.sum(a=h * h_star).real:.6e}  (matches sum |H|^2)")
print(f"sum H H*:  {np.sum(a=H * H_star).real:.6e}  (matches sum |h|^2)")

## Rotation

Rotating the angular samples of the transform by $q_0$ positions rotates the function by the same number of positions [[1]](#ref-1) (Eq. 75), with both indices taken modulo $N_2$:

$$F_{(q - q_0) m} \;\Longleftrightarrow\; f_{(p - q_0) k}.$$

A rotation by $q_0$ angular samples is a rotation by the angle $2\pi q_0 / N_2$. The figure transforms an off-center Gaussian, rotates its transform with `np.roll` along `pypft.Axis.ANGULAR`, and transforms back: the result is the rotated Gaussian. The right-hand panel is the difference from rolling the Gaussian itself, drawn on its own color scale: it is rounding noise, as the printed maximum shows.

In [ ]:
# A larger grid, and an off-center Gaussian at radius 3 and angle 0
rotation_grid = pypft.PolarGrid(n_radial=128, n_angular=31, R=8.0)
r = rotation_grid.r.T
theta = rotation_grid.theta[np.newaxis, :]
f = np.exp(-(r**2 + 9 - 6 * r * np.cos(theta)) / 2)

# Rotate the transform by q0 angular samples, then transform back
q0 = 5
F = pypft.forward_pft(f=f, grid=rotation_grid)
rotated = pypft.inverse_pft(
    F=np.roll(a=F, shift=q0, axis=pypft.Axis.ANGULAR), grid=rotation_grid
)

# The same as rotating the Gaussian itself
expected = np.roll(a=f, shift=q0, axis=pypft.Axis.ANGULAR)
print(
    f"rotation by {q0} samples = {np.degrees(2 * np.pi * q0 / rotation_grid.n_angular):.1f} degrees"
)
print(f"max difference: {np.abs(rotated - expected).max():.1e}")

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 4.5))
panels = (
    (f, "Gaussian"),
    (rotated, "transform rotated, transformed back"),
    (rotated - expected, "difference (own color scale)"),
)
for ax, (values, title) in zip(axes, panels):
    signal = pypft.PolarSpatialSignal(values=values, grid=rotation_grid)
    pypft.render_cartesian(signal=signal, height=300, width=300, ax=ax)
    ax.set_title(label=title)
fig.tight_layout()
plt.show()

## Linearity and the zeroth harmonic

Both transforms are linear: $a f_1 + b f_2 \Leftrightarrow a F_1 + b F_2$.

A function with no angular dependence, the same values in every angular column, has only a zeroth harmonic. The angular DFT then leaves a single column of the harmonic table, the transform of order $0$, and the whole PFT reduces to one order-0 Hankel transform scaled by $2\pi$, the factor $2\pi\, i^{-n}$ at $n = 0$. Every angular column of $F$ is that same transform, the counterpart of the continuous fact that the 2-D Fourier transform of a circularly symmetric function is its order-0 Hankel transform times $2\pi$.

In [ ]:
# Linearity, with two random signals and two coefficients
f1 = rng.standard_normal(size=(grid.n_radial, grid.n_angular))
f2 = rng.standard_normal(size=(grid.n_radial, grid.n_angular))
a, b = 2.3, -1.7
combination = pypft.forward_pft(f=a * f1 + b * f2, grid=grid)
separate = a * pypft.forward_pft(f=f1, grid=grid) + b * pypft.forward_pft(
    f=f2, grid=grid
)
print(f"linearity: max difference {np.abs(combination - separate).max():.1e}")

# A function with no angular dependence: one radial profile in every column
profile = np.exp(-(np.linspace(start=0.0, stop=4.0, num=grid.n_radial) ** 2))
f_symmetric = np.repeat(a=profile[:, np.newaxis], repeats=grid.n_angular, axis=1)
F_symmetric = pypft.forward_pft(f=f_symmetric, grid=grid)

# Every column of F is 2 pi times the order-0 Hankel transform of the profile
hankel = 2 * np.pi * pypft.hankel_transform(f=profile, n=0, R=grid.R)
print(
    f"zeroth harmonic: max difference {np.abs(F_symmetric - hankel[:, np.newaxis]).max():.1e}"
)

# All 31 columns of F lie on top of the Hankel transform
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(F_symmetric.real, color="tab:blue", alpha=0.3)
ax.plot(
    hankel,
    color="black",
    linestyle="--",
    label=r"$2\pi \times$ order-0 Hankel transform",
)
ax.set_xlabel(xlabel="radial index")
ax.set_title(label="real part of every angular column of F")
ax.legend()
plt.show()

## The spectrum of a real signal

The DFT of a real sequence is conjugate symmetric, $f_{-n} = \overline{f_n}$. In the harmonic domain of the PFT, after the Hankel transforms, the symmetry picks up a sign:

$$F_{-n}(\rho) = (-1)^n\, \overline{F_n(\rho)}.$$

The sign comes from the negative orders: harmonic $-n$ is transformed with the kernel $Y^{(-n)N} = (-1)^n Y^{nN}$, while the factors $2\pi\, i^{-n}$ of the two harmonics are complex conjugates of each other. The figure checks it at $n = 3$, at the `POLAR_FREQUENCY_HARMONIC` domain of [04_domains](04_domains.ipynb), for a real random signal.

In [ ]:
# A real random signal, walked to the harmonic frequency domain
f_real = rng.standard_normal(size=(grid.n_radial, grid.n_angular))
spectrum = (
    pypft.PolarSpatialSignal(values=f_real, grid=grid)
    .to(domain=pypft.Domain.POLAR_FREQUENCY_HARMONIC)
    .values
)

# The columns of harmonics n and -n
n = 3
column_positive = spectrum[:, np.flatnonzero(a=grid.harmonics == n)[0]]
column_negative = spectrum[:, np.flatnonzero(a=grid.harmonics == -n)[0]]
twisted = (-1) ** n * np.conj(column_positive)
print(f"max |F_-n - (-1)^n conj(F_n)|: {np.abs(column_negative - twisted).max():.1e}")
print(
    f"max |F_-n - conj(F_n)|:        {np.abs(column_negative - np.conj(column_positive)).max():.1e}"
)

# Real and imaginary parts of both sides
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(11, 3.8))
for ax, part, label in (
    (axes[0], np.real, "real part"),
    (axes[1], np.imag, "imaginary part"),
):
    ax.plot(
        part(column_negative),
        marker="o",
        markersize=3,
        linestyle="none",
        label=f"$F_{{-{n}}}$",
    )
    ax.plot(part(twisted), color="black", label=f"$(-1)^{n}\\,\\overline{{F_{n}}}$")
    ax.set_xlabel(xlabel="radial index")
    ax.set_title(label=label)
    ax.legend()
fig.tight_layout()
plt.show()

## Where to go next

[08_visualization](08_visualization.ipynb) traces a transform through every domain and saves the figures.

## References

1. <a id="ref-1"></a> N. Baddour, "Discrete Two-Dimensional Fourier Transform in Polar Coordinates Part I: Theory and Operational Rules," *Mathematics*, 7(8):698, 2019. [doi:10.3390/math7080698](https://doi.org/10.3390/math7080698)